In [1]:
!pip install evaluate rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=efd3fa25881466b34658b82930c85e95888b674a9cd68465e6a677e0c8c6a765
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
import evaluate
import numpy as np

print("Завантаження інструментів...")
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

references = [
    "КМУ скасував часові обмеження щодо увімкнення денних ходових вогнів... Відтепер ця вимога діє цілий рік для всіх транспортних засобів.",
    "Споживач має право обміняти непродовольчий товар належної якості протягом 14 днів з моменту покупки (не рахуючи день купівлі). Обов'язковими умовами для обміну є збереження товарного вигляду, пломб, ярликів та наявність розрахункового документа, а також факт невикористання товару.",
    "Крадіжка (таємне викрадення чужого майна) карається штрафом, громадськими або виправними роботами, арештом до 6 місяців або обмеженням волі до 5 років. При повторному вчиненні або вчиненні групою осіб покарання посилюється до позбавлення волі строком до 5 років.",
    "Мінімальна тривалість щорічної основної відпустки становить 24 дні за відпрацьований рік. Під час дії воєнного стану роботодавець має законне право обмежити загальну тривалість відпустки саме цим строком.",
    "Податкова знижка дозволяє фізичним особам-резидентам зменшити свій річний оподатковуваний дохід (заробітну плату) на суму документально підтверджених витрат на товари та послуги, придбані у резидентів протягом звітного року.",
    "Аліменти призначаються судом як частка від доходу або у фіксованій сумі за вибором опікуна, з яким проживає дитина. Їх мінімальний розмір гарантовано становить щонайменше 50% прожиткового мінімуму для дитини відповідного віку.",
    "Електронний договір укладається через процедуру оферти та акцепту. Юридично він вважається укладеним і набуває чинності з моменту, коли ініціатор отримує підтвердження про прийняття його пропозиції іншою стороною.",
    "Військовозобов'язані, які офіційно заброньовані державними органами, органами місцевого самоврядування або визначеними підприємствами відповідно до порядку КМУ, звільняються від призову під час мобілізації.",
    "Державною мовою (українською) здійснюється весь освітній процес в Україні. Держава гарантує громадянам безперешкодне право здобувати освіту будь-якого рівня державною мовою у всіх державних та комунальних закладах.",
    "Договір оренди будівлі завжди укладається письмово. Обов'язковому нотаріальному посвідченню підлягають договори зі строком оренди від 3 років (для державного чи комунального майна на аукціоні — зі строком понад 5 років)."
]

predictions = [
    "Поза населених пунктів денні ходові вогні або ближнє світло фар обов'язково мають бути включені на всіх транспортних засобах під час руху у світлу пору доби. Це правило діє цілий рік, а не лише в період з 1 жовтня по 1 травня.",
    "Споживач має право обміняти непридатний товар належної якості на аналогічний протягом 14 днів після купівлі, якщо це не визначено довшою строком продавцем. Обмін здійснюється лише у тому магазині, де було придбано товари.",
    "За викрадення чужого майна можуть застосувати штраф, громадські роботи, виправні роботи або арешт. За повторне чи групове викрадення — позбавлення волі.",
    "За відпрацювання робочого року працівник має право на щорічну відпустку тривалістю не менше 24 днів. У воєнний час тривалість відпустки може бути обмежена національним законодавством.",
    "Податкова знижка для громадян — це сума витрат на придбання товарів або послуг у резидентів, яка зменшує податковий наклад на дохід заробітної плати.",
    "Аліменти на дитину присуджуються за рішенням суду з урахуванням доходу батька та потреб дитини. Мінімальний розмір аліментів визначено законом і не може бути меншим, ніж 50% від прожиткового мінімуму для дітей.",
    "Електронний договір укладається шляхом направлення пропозиції (оферти) однією стороною та її прийняття (акцепту) другою стороною. Договір вважається укладеним з моменту отримання відповіді на пропозицію.",
    "Заброньовані військовозобов'язані не підлягають призову на військову службу під час мобілізації. Це підприємства, установи та організації, що важливі для забезпечення обороноздатності та життєдіяльності населення.",
    "Державна мова є офіційною мовою навчання в українських закладах освіти на будь-якому рівні. Кожному громадянинові гарантується право на здобуття освіти державною мовою в державних та комунальних закладах.",
    "Договір найму будівлі або іншої капітальної споруди укладається письмово та підлягає нотаріальному посвідченню, якщо строк дії договору більше трьох років. Договір майнових відносин з державною або комунальною власністю також може підлягати нотаріальному посвідченню."
]

print("Рахуємо ROUGE (Лексичний збіг)...")
rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    tokenizer=lambda x: x.split()
)

print("Рахуємо BERTScore (Смисловий збіг)...")
bert_results = bertscore.compute(predictions=predictions, references=references, lang="uk")
avg_bert_f1 = np.mean(bert_results['f1'])

print("\n" + "="*50)
print("РЕЗУЛЬТАТИ ОЦІНКИ)")
print("="*50)
print(f"ROUGE-1 (Збіг слів):         {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 (Збіг словосполучень): {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L (Структура речень):  {rouge_results['rougeL']:.4f}")
print("-" * 50)
print(f"BERTScore F1 (Смисловий збіг): {avg_bert_f1:.4f}")
print("="*50)

Завантаження інструментів...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Рахуємо ROUGE (Лексичний збіг)...
Рахуємо BERTScore (Смисловий збіг)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



РЕЗУЛЬТАТИ ОЦІНКИ)
ROUGE-1 (Збіг слів):         0.2420
ROUGE-2 (Збіг словосполучень): 0.0868
ROUGE-L (Структура речень):  0.2015
--------------------------------------------------
BERTScore F1 (Смисловий збіг): 0.7732


Порівняємо результати Експерименту 2 та Експерименту 3:

BERTScore (Смисл): Було 0.7733 ➡️ Стало 0.7732.

ROUGE-1 (Слова): Було 0.2364 ➡️ Стало 0.2420.

---
модель детермінована за змістом, але варіативна за формою. Оскільки ROUGE трохи підріс (модель підібрала інші синоніми та по-іншому побудувала речення), лексичний збіг змінився. Але семантична нейромережа BERT побачила, що юридична суть залишилася ідентичною (77.3%).
Для LegalTech це найважливіший показник: юристу не так важливо, якими саме словами написане саммарі, головне — щоб сенс закону не спотворювався від запуску до запуску.